# colab

> Jobs on Colab VMs through Google's `colab` CLI: a CPU, a T4, L4, G4, A100 or H100 GPU, or a v5e-1 or v6e-1 TPU; the job runs detached on the VM, short calls follow its log, and the VM is released as soon as the job ends.

In [ ]:
#| default_exp colab

In [ ]:
#| export
import json, os, re, shlex, shutil, subprocess, sys, tempfile, time
from pathlib import Path

from sysone.auth import refresh_colab_token
from sysone.cloud import (Colab, CPU, GPU, TPU, Job, LocalPath, HubData, machine, stage, submit, poll, cancel, collect,
                          pack, unpack_outputs, save_job, load_jobs)

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail, test_ne
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
from sysone.cloud import remote, jobs, hardware
from sysone.inference import Decider

Google's [Colab CLI](https://github.com/googlecolab/google-colab-cli) (`uv tool install google-colab-cli`, and once `sysone.auth.login("colab")`, which asks Google for your email and Colab only, see [auth](43_auth.ipynb#colab)) creates a VM, uploads files, runs code in the VM's Jupyter kernel and stops the VM. It has no notion of a job, though: `colab run` and `colab exec` hold a connection open for as long as the code runs, end it at a fixed deadline, and lose the run if the connection drops. So a sysone job on Colab runs detached:

1. `colab new` creates the VM, named after the job, with the hardware's flags;
2. `colab upload` sends the job's folder as one `.tar.gz` (the CLI moves single files), and the secrets as a second file, into `/content`, which every VM has (an upload makes no folders);
3. one short `colab exec` unpacks the folder and starts the [runner](40_cloud.ipynb#the-jobs-files) in a background process of the VM, writing `job.log`;
4. every poll is another short `colab exec` that returns the log since the last poll, the GPU's use as `nvidia-smi` reports it (a progress line of its own), and, at the end, the runner's `status.json`;
5. when the job has ended, `colab download` brings `outputs.tar.gz` back and `colab stop` releases the VM (unless `Colab(keep=True)`), so an idle VM does not keep using compute units.

Each job keeps the CLI's session state in a file of its own (`--config`), which lets several jobs run at once. The CLI writes the code of every `colab exec` to a history file under `~/.config/colab-cli/history`, so nothing secret goes into that code: the Hub token (for pushing the export or reading a private dataset) travels as an uploaded file, which the runner reads and deletes.

::: {.callout-note title="Finding: the Colab CLI writes the code of every `colab exec`, its `--env` values included, to a local history file"}
`colab exec --env KEY=VALUE` sets the variable by putting `os.environ['KEY'] = 'VALUE'` in front of the code it runs, and the CLI appends the code of every execution to `~/.config/colab-cli/history/<session>.jsonl`. A Hub token passed that way would sit there in plain text. Uploads are logged by path only, so sysone sends secrets as an uploaded file, which the runner reads and deletes on the VM; the stand-in below records every command line and every piece of code it is given, and the token is in neither.
:::

In [ ]:
#| export
class ColabError(RuntimeError): "The Colab CLI reported a failure"

COLAB_ROOT = "/content/sysone_jobs"
COLAB_INBOX = "/content"          # where uploads land: Colab's file API writes into folders that exist, and makes none
COLAB_GPUS = ("T4", "L4", "G4", "A100", "H100")
COLAB_TPUS = {"v5e-1": "v5e1", "v6e-1": "v6e1"}

def colab_cmd(b:Colab, job:Job|None, *args) -> list:
    "A `colab` command line with the backend's sign-in and, for a job, its own session-state file"
    cfg = ["--config", str(Path(job.folder) / "colab-sessions.json")] if job is not None and job.folder else []
    return ["colab", f"--auth={b.auth}"] + cfg + [str(a) for a in args]

def run_colab(cmd:list, input:str|None=None, timeout:float=600, check:bool=True) -> str:
    "Run a `colab` command and return its standard output (renewing sysone's Colab-only token first, see auth)"
    if shutil.which(cmd[0]) is None: raise ColabError("the Colab CLI is not installed: uv tool install google-colab-cli")
    try: refresh_colab_token()
    except Exception as e: print(f"sysone: could not renew the Colab token ({e}); the CLI will try", file=sys.stderr)
    try: r = subprocess.run(cmd, input=input, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: raise ColabError(f"{' '.join(cmd[:4])} took over {timeout:.0f} s") from None
    if check and r.returncode != 0:
        raise ColabError(f"{' '.join(c for c in cmd if not c.startswith('--config'))} failed ({r.returncode}): {(r.stdout + r.stderr).strip()[-1500:]}")
    return r.stdout

## Hardware and data

A Colab VM has one GPU (T4, L4, G4, A100 or H100), a TPU (v5e-1 or v6e-1), or neither. The CLI quietly turns a GPU name it does not know into an A100, so the name is checked here first. The free tier gets a T4 and a v5e-1 TPU; the others need compute units. Local data travels with the job up to `COLAB_SHIP_MB` (the CLI uploads it whole, in one request); larger data goes through a private Hub dataset (`sysone.hub.push_data`, then `hf:owner/name`).

In [ ]:
#| export
COLAB_SHIP_MB = 100

@machine.register
def machine_colab_cpu(b:Colab, hw:CPU) -> str: return "--high-mem" if hw.high_mem else "cpu"

@machine.register
def machine_colab_gpu(b:Colab, hw:GPU) -> str:
    if hw.name not in COLAB_GPUS or hw.count != 1: raise ValueError(f"a Colab VM has one GPU of {', '.join(COLAB_GPUS)}, not {hw}")
    return f"--gpu {hw.name}" + (" --high-mem" if hw.high_mem else "")

@machine.register
def machine_colab_tpu(b:Colab, hw:TPU) -> str:
    if hw.kind not in COLAB_TPUS: raise ValueError(f"Colab's TPUs are {', '.join(COLAB_TPUS)}, not {hw.kind}")
    return f"--tpu {COLAB_TPUS[hw.kind]}"

def _size(p:Path) -> int: return p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

@stage.register
def stage_colab_local(d:LocalPath, b:Colab, var:str) -> dict:
    mb = _size(Path(d.path)) / 2**20
    if mb > COLAB_SHIP_MB: raise ValueError(f"{d.path} is {mb:.0f} MB: push it to the Hub (sysone.hub.push_data) and pass hf:owner/name")
    return {"ship": {var: d.path}}

In [ ]:
test_eq([machine(Colab(), h) for h in (CPU(), GPU("t4"), GPU("A100", high_mem=True), TPU("v6e-1"))],
        ["cpu", "--gpu T4", "--gpu A100 --high-mem", "--tpu v6e1"])
test_fail(lambda: machine(Colab(), GPU("T4", 2)), contains="one GPU")
test_fail(lambda: machine(Colab(), TPU("v5e-8")), contains="Colab's TPUs")
test_eq(stage(LocalPath(str(Path("fixtures/typed_decisions_tiny.jsonl").resolve())), Colab(), "SYSONE_DATA")["ship"]["SYSONE_DATA"].endswith("tiny.jsonl"), True)

## Secrets

`Colab(secrets=("HF_TOKEN",))` names the variables a job gets from this machine: each one's value in the environment, and for `HF_TOKEN` the Hub login of this machine when the variable is unset. Only the names that have a value are sent.

In [ ]:
#| export
def job_secrets(b:Colab) -> dict:
    "The values of the backend's secrets on this machine (the Hub login for HF_TOKEN), for the names that have one"
    out = {}
    for name in b.secrets:
        v = os.environ.get(name)
        if v is None and name == "HF_TOKEN":
            from huggingface_hub import get_token
            v = get_token()
        if v: out[name] = v
    return out

## Starting and following a job

In [ ]:
#| export
def _flags(machine:str) -> list: return [] if machine == "cpu" else shlex.split(machine)

def start_code(job:Job) -> str:
    "What the one `colab exec` that starts a job runs: unpack the job's folder and start its runner in the background"
    d, tar = f"{COLAB_ROOT}/{job.id}", f"{COLAB_INBOX}/{job.id}.tar.gz"
    return f"""import os, shutil, subprocess, sys, tarfile
os.makedirs({d!r}, exist_ok=True)
with tarfile.open({tar!r}) as t: t.extractall({d!r}, filter="data")
os.remove({tar!r})
if os.path.exists({tar + '.secrets'!r}): shutil.move({tar + '.secrets'!r}, os.path.join({d!r}, ".secrets.json"))
log = open(os.path.join({d!r}, "job.log"), "ab")
p = subprocess.Popen([sys.executable, os.path.join({d!r}, "sysone_job.py")], stdout=log, stderr=subprocess.STDOUT, cwd={d!r}, start_new_session=True)
print("SYSONE-PID", p.pid)
"""

def poll_code(job:Job, max_bytes:int=500_000) -> str:
    "What a poll's `colab exec` runs: the log from the job's offset, the runner's status when it has ended, and the GPU's use"
    d = f"{COLAB_ROOT}/{job.id}"
    return f"""import json, os, subprocess
log, st, txt = os.path.join({d!r}, "job.log"), os.path.join({d!r}, "outputs", "status.json"), b""
if os.path.exists(log):
    with open(log, "rb") as f: f.seek({job.offset}); txt = f.read({max_bytes})
gpu = None
try: gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout.strip()
except Exception: pass
done = os.path.exists(st) and os.path.exists(os.path.join({d!r}, "outputs.tar.gz"))
print("SYSONE-POLL " + json.dumps({{"text": txt.decode("utf-8", "replace"), "n": len(txt), "status": json.load(open(st)) if done else None, "gpu": gpu}}))
"""

@submit.register
def submit_colab(b:Colab, job:Job) -> Job:
    folder = Path(job.folder)
    tar = folder.parent / f"{job.id}.tar.gz"; tar.write_bytes(pack(folder, skip=()))
    run_colab(colab_cmd(b, job, "new", "-s", job.id, *_flags(job.machine)), timeout=900)
    job.ref, job.status = job.id, "starting"; save_job(job)
    try:
        run_colab(colab_cmd(b, job, "upload", "-s", job.id, tar, f"{COLAB_INBOX}/{job.id}.tar.gz"))
        sec = job_secrets(b)
        if sec:
            fd, sp = tempfile.mkstemp(suffix=".json")
            try:
                with os.fdopen(fd, "w") as f: json.dump(sec, f)
                run_colab(colab_cmd(b, job, "upload", "-s", job.id, sp, f"{COLAB_INBOX}/{job.id}.tar.gz.secrets"))
            finally: os.remove(sp)
        out = run_colab(colab_cmd(b, job, "exec", "-s", job.id, "--timeout", "300"), input=start_code(job))
        if "SYSONE-PID" not in out: raise ColabError(f"the job did not start: {out[-500:]}")
    except Exception:
        run_colab(colab_cmd(b, job, "stop", "-s", job.id), check=False)     # a VM that runs nothing is released at once
        job.status = "error"; save_job(job)
        raise
    finally: tar.unlink(missing_ok=True)
    job.status = "running"
    return job

@poll.register
def poll_colab(b:Colab, job:Job) -> str:
    out = run_colab(colab_cmd(b, job, "exec", "-s", job.id, "--timeout", "120"), input=poll_code(job), timeout=240, check=False)   # see the finding below
    line = next((l for l in out.splitlines() if l.startswith("SYSONE-POLL ")), None)
    if line is None:
        if re.search(r"not found|404|401|lost", out, re.I): job.status = "error"
        return ""
    r = json.loads(line[len("SYSONE-POLL "):])
    job.offset += r["n"]
    text = r["text"]
    if r.get("gpu"):                     # the VM's GPU, as nvidia-smi sees it: a progress line of its own
        try:
            util, used, total = (float(x) for x in r["gpu"].splitlines()[0].split(","))
            text += "SYSONE " + json.dumps({"t": round(time.time(), 1), "event": "vm", "gpu_util": util,
                                            "gpu_mem_gb": round(used / 1024, 2), "gpu_total_gb": round(total / 1024, 2)}) + "\n"
        except ValueError: pass
    with open(Path(job.folder) / "job.log", "a") as f: f.write(text)
    if r["status"] is not None:          # ended: bring the outputs back, then release the VM
        run_colab(colab_cmd(b, job, "download", "-s", job.id, f"{COLAB_ROOT}/{job.id}/outputs.tar.gz", Path(job.folder) / "outputs.tar.gz"))
        if not b.keep: run_colab(colab_cmd(b, job, "stop", "-s", job.id), check=False)
        job.status, job.url = r["status"]["status"], r["status"].get("url") or job.url
    return text

@cancel.register
def cancel_colab(b:Colab, job:Job) -> Job:
    run_colab(colab_cmd(b, job, "stop", "-s", job.id), check=False)
    job.status = "stopped"
    return job

@collect.register
def collect_colab(b:Colab, job:Job, dest:Path) -> Path:
    tb = Path(job.folder) / "outputs.tar.gz"
    if not tb.exists():
        if job.status in ("stopped", "error") and not tb.exists(): raise ColabError(f"{job.id} ended {job.status}: there are no outputs to bring back")
        raise ColabError(f"{job.id} is {job.status}: wait for it to end (job.wait())")
    return unpack_outputs(tb, dest)

def colab_usage() -> str:
    "Your Colab compute units: balance and rate (`colab usage`)"
    return run_colab(["colab", "--auth=oauth2", "usage"])

::: {.callout-note title="Finding: a `colab exec` can outlive its own timeout"}
On a real run, one poll of a CPU VM that was busy training stayed connected to Colab, waiting for a reply, for over two minutes, past its own `--timeout 120`. Once it was stopped, the next poll went through and the job ended normally. So a poll gives up after 240 s, and `wait` takes a failed poll in its stride and polls again, up to five failures in a row (see [cloud](40_cloud.ipynb)).
:::

::: {.callout-note title="Finding: a poll fetched the outputs while they were being packed"}
A poll takes a job as ended once the runner's `status.json` and `outputs.tar.gz` are both there, and the runner wrote the tarball in place. On a TPU job, whose export holds a 59 MB head, a poll found the tarball after 773 KB and brought back a truncated archive. The runner now packs into `outputs.tar.gz.part` and renames it when it is whole, so a poll sees the whole tarball or none.
:::

## A job on a stand-in Colab

The whole round trip runs here against a stand-in for the `colab` CLI that keeps a VM's files in a temporary folder (`/content` mapped there) and runs `exec` code with this Python. It records every command line and every piece of code it is given, which shows that the token reached the job without passing through either:

In [ ]:
fake_bin, fake_vm = Path(tempfile.mkdtemp()), Path(tempfile.mkdtemp())
(fake_bin / "colab").write_text(f"""#!{sys.executable}
import json, os, shutil, subprocess, sys
root = os.environ["FAKE_COLAB_ROOT"]
args = [a for a in sys.argv[1:] if not a.startswith("--auth")]
if "--config" in args: i = args.index("--config"); del args[i:i + 2]
cmd, rest, opts, pos = args[0], args[1:], {{}}, []
it = iter(rest)
for a in it:
    if a in ("-s", "--timeout", "--gpu", "--tpu", "--env"): opts[a] = next(it)
    elif a.startswith("--"): opts[a] = True
    else: pos.append(a)
vm = lambda p: os.path.join(root, p.lstrip("/"))
open(os.path.join(root, "calls.log"), "a").write(json.dumps(sys.argv[1:]) + "\\n")
if cmd == "new": os.makedirs(vm("/content"), exist_ok=True); print("[colab] ready")
elif cmd == "upload":                  # as Colab's file API: into a folder that exists, or not at all
    if not os.path.isdir(os.path.dirname(vm(pos[1]))): print("[colab] Upload failed: 500 Server Error"); sys.exit(1)
    shutil.copy(pos[0], vm(pos[1]))
elif cmd == "download": shutil.copy(vm(pos[0]), pos[1])
elif cmd == "exec":
    code = sys.stdin.read(); open(os.path.join(root, "exec.log"), "a").write(code + "\\n----\\n")
    sys.exit(subprocess.run([sys.executable, "-c", code.replace("/content", vm("/content"))]).returncode)
elif cmd == "stop": print("[colab] stopped")
""")
(fake_bin / "colab").chmod(0o755)
(fake_bin / "nvidia-smi").write_text("#!/bin/sh\necho '37, 2150, 15360'\n"); (fake_bin / "nvidia-smi").chmod(0o755)   # the VM's GPU, as a T4 would report it
os.environ |= {"PATH": f"{fake_bin}{os.pathsep}{os.environ['PATH']}", "FAKE_COLAB_ROOT": str(fake_vm), "HF_TOKEN": "hf_not_a_real_token",
               "SYSONE_COLAB_TOKEN": str(Path(tempfile.mkdtemp()) / "no-token.json"),      # never your real Colab token
               "SYSONE_JOBS": str(Path(tempfile.mkdtemp()) / "jobs.json"), "SYSONE_JOBS_DIR": tempfile.mkdtemp()}

In [ ]:
script = Path(tempfile.mkdtemp()) / "train.py"
script.write_text("""import os
from sysone.data import TypedDecisions
from sysone.learner import decision_learner
from sysone.cloud import job_input, job_output
data = TypedDecisions.from_json(job_input("data"), valid=0.25, calib=0.2)
learn = decision_learner(data, os.environ["ENCODER"], train="head", logging_steps=1, disable_tqdm=True)
learn.fit(1, lr=1e-3)
learn.export(job_output())
open(os.path.join(os.path.dirname(job_output()), "token_seen"), "w").write(str(os.environ.get("HF_TOKEN") == "hf_not_a_real_token"))
""")
cj = remote(script, on="colab", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": str(Path("fixtures/tiny-modernbert").resolve())},
            install=False)      # the stand-in VM is this machine: nothing to install
cj.wait(every=2, logs=False)
test_eq(cj.status, "complete")
exp = cj.fetch()
test_eq((Path(cj.outputs) / "token_seen").read_text(), "True")
calls, code = (fake_vm / "calls.log").read_text(), (fake_vm / "exec.log").read_text()
assert "hf_not_a_real_token" not in calls and "hf_not_a_real_token" not in code
cmds = [json.loads(l)[3] for l in calls.splitlines()]          # after --auth and the job's own --config
test_eq(cmds[:3], ["new", "upload", "upload"])                   # the VM, the job, the secrets
test_eq(cmds[-2:], ["download", "stop"])                         # the outputs, then the VM released
assert [r for r in cj.progress() if r["event"] == "log"]                                       # the progress lines came back with the polls
vm = [r for r in cj.progress() if r["event"] == "vm"]
test_eq((vm[0]["gpu_util"], vm[0]["gpu_total_gb"]), (37.0, 15.0))                              # and the VM's GPU use with them

In [ ]:
rec = json.loads(Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()[0])
test_eq(set(Decider.load(exp, device="cpu").predict(rec["state"], rec["questions"])), set(rec["questions"]))
jobs(backend="colab")

In [ ]:
os.environ["PATH"] = os.environ["PATH"].split(os.pathsep, 1)[1]
del os.environ["FAKE_COLAB_ROOT"], os.environ["HF_TOKEN"], os.environ["SYSONE_COLAB_TOKEN"]     # the real cells below use your token

## On a real Colab VM

These cells need the Colab CLI and a login (`sysone.auth.login("colab")`); they are flagged `colab`, so the tests skip them unless run with `--flags colab`. The first runs the tiny training on a CPU VM, which the free tier allows; the second does it on a T4.

In [ ]:
#| colab
real = remote(script, on="colab", hw="cpu", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"})
real.wait(every=20)
test_eq(real.status, "complete")
Decider.load(real.fetch(), device="cpu").predict(rec["state"], rec["questions"])

In [ ]:
#| colab
t4 = remote(script, on="colab", hw="t4", data="fixtures/typed_decisions_tiny.jsonl", env={"ENCODER": "answerdotai/ModernBERT-base"})
t4.wait(every=20)
test_eq(t4.status, "complete")
[r for r in t4.progress() if r["event"] == "log"][-1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()